In [3]:
#Operative system
import sys
from pathlib import Path

#Script importation
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    src_path = cwd.parent / "src"
else:
    src_path = cwd / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
import graphon as graphon
import metrics as metrics

#Libraries
import torch
from torch_geometric.data import Data
from torch_geometric.utils import degree

#Autoreload module
"""
These magic lines allow to automatically reload any module.py file when there are some changes on it, without the needing of restart the kernel anytime a new change is applied to the module script
"""
%load_ext autoreload
%autoreload 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/Users/luca.frattegiani/software/miniconda3/envs/graphs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Generate some random W-graphs
modes = ["ring", "sbm", "random"] #Types of graphs
n_graphs = 5 #Number of graphs to generate for each mode
device = "cpu"
n = 50 #Number of nodes in each graph
K = 5 #Discrete state space
graph_set = {mode : None for mode in modes}
for mode in graph_set.keys():
    graphs = [None] * n_graphs
    z = torch.rand(n, device = device) #Latent positions
    z = torch.sort(z).values #Node 0 has the smallest latent position

    # Edges (undirected)
    start, end = torch.triu_indices(n, n, offset = 1, device = device) #All possible combinations of undirected links
    z_start = z[start] #Latent positions of source nodes
    z_end = z[end] #Latent positions of ending nodes

    for i in range(n_graphs):
        if mode == "ring":
            probs = graphon.ring_graphon(z_start, z_end)
        elif mode == "sbm":
            blocks = torch.tensor([[0.8, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]], device = "cpu")
            probs = graphon.sbm_graphon(z_start, z_end, blocks)
        elif mode == "random":
            p = 0.5
            probs = torch.full_like(z_start, p, device = "cpu")
    
        # Sample links with inverse transform
        u = torch.rand_like(probs) #Uniform distributions over [0, 1]
        links = u < probs #Bernoulli samples

        # Keep only existing edges
        edge_start = start[links]
        edge_end = end[links]

        # Create edge set
        edges = torch.cat(
            [
                torch.stack([edge_start, edge_end]), #Connection i->j
                torch.stack([edge_end, edge_start]), #Connection j->i
            ],
            dim = 1
        )

        # Sample node features
        x = torch.randint(low = 0, high = K, size = (n, 1), device = device)

        # Create Data pytorch object
        graph = Data(
            x = x,
            edge_index = edges,
            num_nodes = n,
        )

        graphs[i] = graph

    graph_set[mode] = graphs

In [5]:
#Density and clustering coefficients
for mode in graph_set.keys():
    print(f"{mode} graphs:")
    densities = [metrics.density(graph) for graph in graph_set[mode]]
    loc_clust = [metrics.local_clustering_coeff(graph) for graph in graph_set[mode]]
    global_clust = [metrics.global_clustering_coeff(graph) for graph in graph_set[mode]]
    print(f"Density: {densities}")
    #print(f"Local: {loc_clust}")
    print(f"Global clustering: {global_clust}")
    print("="*50)

ring graphs:
Density: [0.19428571428571428, 0.19591836734693877, 0.18857142857142858, 0.18693877551020407, 0.19020408163265307]
Global clustering: [tensor(0.5313), tensor(0.5477), tensor(0.5665), tensor(0.5489), tensor(0.5387)]
sbm graphs:
Density: [0.3526530612244898, 0.3526530612244898, 0.36081632653061224, 0.3436734693877551, 0.34530612244897957]
Global clustering: [tensor(0.6096), tensor(0.6280), tensor(0.6170), tensor(0.5731), tensor(0.6352)]
random graphs:
Density: [0.49387755102040815, 0.4816326530612245, 0.5126530612244898, 0.4930612244897959, 0.4963265306122449]
Global clustering: [tensor(0.4904), tensor(0.4826), tensor(0.5103), tensor(0.4950), tensor(0.4963)]


In [6]:
#Centrality coefficients
for mode in graph_set.keys():
    print(f"{mode} graphs:")
    harmonic_cent = [metrics.harmonic_centrality(graph, centralization = "freeman") for graph in graph_set[mode]]
    betweenness_cent = [metrics.betweenness_centrality(graph, centralization = "freeman") for graph in graph_set[mode]]
    pagerank_cent = [metrics.pagerank_centrality(graph, centralization = "entropy") for graph in graph_set[mode]]
    print(f"Harmonic: {harmonic_cent}")
    print(f"Betweenness: {betweenness_cent}")
    print(f"Pagerank: {pagerank_cent}")
    print("="*50)

ring graphs:
Harmonic: [tensor(0.1586), tensor(0.1633), tensor(0.1549), tensor(0.1662), tensor(0.1135)]
Betweenness: [tensor(0.0970), tensor(0.0609), tensor(0.1118), tensor(0.0672), tensor(0.0756)]
Pagerank: [tensor(0.0038), tensor(0.0024), tensor(0.0055), tensor(0.0030), tensor(0.0033)]
sbm graphs:
Harmonic: [tensor(0.1956), tensor(0.1811), tensor(0.1845), tensor(0.1808), tensor(0.2245)]
Betweenness: [tensor(0.0316), tensor(0.0277), tensor(0.0205), tensor(0.0276), tensor(0.0265)]
Pagerank: [tensor(0.0045), tensor(0.0035), tensor(0.0055), tensor(0.0041), tensor(0.0044)]
random graphs:
Harmonic: [tensor(0.1446), tensor(0.1361), tensor(0.1250), tensor(0.2092), tensor(0.2483)]
Betweenness: [tensor(0.0083), tensor(0.0075), tensor(0.0052), tensor(0.0114), tensor(0.0149)]
Pagerank: [tensor(0.0017), tensor(0.0021), tensor(0.0014), tensor(0.0024), tensor(0.0023)]


In [7]:
#Homophily coefficients
for mode in graph_set.keys():
    print(f"{mode} graphs:")
    jsd_informativity = [metrics.jsd_informativity(graph) for graph in graph_set[mode]]
    adj_homophily = [metrics.adjusted_homophily(graph) for graph in graph_set[mode]]
    print(f"JSD: {jsd_informativity}")
    print(f"Adjusted: {adj_homophily}")
    print("="*50)

ring graphs:
JSD: [tensor(0.0102), tensor(0.0093), tensor(0.0089), tensor(0.0074), tensor(0.0049)]
Adjusted: [tensor(-0.0793), tensor(-0.0844), tensor(-0.0526), tensor(-0.0406), tensor(-0.0396)]
sbm graphs:
JSD: [tensor(0.0040), tensor(0.0064), tensor(0.0035), tensor(0.0053), tensor(0.0068)]
Adjusted: [tensor(-0.0424), tensor(0.0074), tensor(-0.0013), tensor(-0.0291), tensor(-0.0200)]
random graphs:
JSD: [tensor(0.0024), tensor(0.0017), tensor(0.0027), tensor(0.0010), tensor(0.0025)]
Adjusted: [tensor(-0.0027), tensor(-0.0129), tensor(-0.0286), tensor(-0.0232), tensor(-0.0296)]
